# Step on a Platinum (211) surface

## 0. Introduction

This notebook demonstrates how to create a slab of Pt(211) which will show the step from the manuscript:

> **Šljivančanin, Ž., & Hammer, B.**
> "Oxygen dissociation at close-packed Pt terraces, Pt steps, and Ag-covered Pt steps studied with density functional theory."
> Surface Science, 515(1), 235–244. 
> [DOI: 10.1016/s0039-6028(02)01908-8 ](https://doi:10.1016/s0039-6028(02)01908-8 )

Replicating the material from the FIG. 1. B. The notebook also puts the O₂ precursor and the dissociated O pair on flat Pt(111) (FIG. 1. A) and at the Pt(211) step, and saves the four materials the [simulation notebook](defect_surface_step_platinum_SIMULATION.ipynb) loads.

<img src="https://github.com/Exabyte-io/documentation/raw/12617167278ae3523adc028583b21ea4e8ebd197/images/tutorials/materials/defects/defect_surface_step_platinum/0-figure-from-manuscript.webp" alt="Pt(211) step" width="400"/>




## 1. Prepare the Environment
### 1.1. Set up defect parameters 

In [ ]:
MATERIAL_NAME = "Pt"
MILLER_INDICES = (2, 1, 1)
THICKNESS = 2  # in (211) slab layers; one layer is four atomic planes
VACUUM = 10.0  # in angstroms
XY_SUPERCELL_MATRIX = [[1, 0], [0, 1]]
USE_ORTHOGONAL_C = True
USE_CONVENTIONAL_CELL = True

ADSORBATE_ELEMENT = "O"
# O-O distance of the adsorbed O2 precursor, Šljivančanin & Hammer 2002, Table 1, in angstroms
PRECURSOR_BOND_LENGTH_111 = 1.36  # Pt(111), top-fcc-bridge
PRECURSOR_BOND_LENGTH_211 = 1.37  # Pt(211) step, top-bridge-top
# in crystal units; snapped to the nearest hollow (fcc on the 3-layer slab)
PRECURSOR_SITE_COORDINATE_111 = [0.5, 0.5]
# starting height of every O above the topmost Pt for the relaxation, in angstroms
ADSORBATE_HEIGHT = 1.8

VISUALIZATION_REPETITIONS = [1, 1, 1]

### 1.2. Install Packages
The step executes only in Pyodide environment. For other environments, the packages should be installed via `pip install` (see [README](../../README.ipynb)).

In [ ]:
from mat3ra.notebooks_utils.packages import install_packages

await install_packages("made|specific_examples")

### 1.3. Get input materials

In [ ]:
from mat3ra.made.tools.build import MaterialWithBuildMetadata
from mat3ra.standata.materials import Materials

material = MaterialWithBuildMetadata.create(Materials.get_by_name_first_match(MATERIAL_NAME))

### 1.4. Preview the material

In [ ]:
from mat3ra.notebooks_utils.ipython.entity.material.visualize import visualize_materials as visualize

visualize(material, repetitions=VISUALIZATION_REPETITIONS, rotation="0x")
visualize(material, repetitions=VISUALIZATION_REPETITIONS, rotation="-90x")

## 2. Create a step with a slab

### 2.1. Create slab configuration
Slab Configuration lets define the slab thickness, vacuum, and the Miller indices of the interfacial plane and get the slabs with possible terminations.


In [ ]:
from mat3ra.made.tools.helpers import create_slab

slab_211 = create_slab(
    crystal=material,
    miller_indices=MILLER_INDICES,
    number_of_layers=THICKNESS,
    vacuum=VACUUM,
    use_orthogonal_c=USE_ORTHOGONAL_C,
    use_conventional_cell=USE_CONVENTIONAL_CELL,
    xy_supercell_matrix=XY_SUPERCELL_MATRIX
)

### 2.2. Visualize the slab

In [ ]:
visualize(slab_211, repetitions=VISUALIZATION_REPETITIONS, rotation="0x")
visualize(slab_211, repetitions=VISUALIZATION_REPETITIONS, rotation="-90x")

## 3. Create a step with terrace builder
### 3.1. Setup terrace parameters

In [ ]:
from types import SimpleNamespace
# Material selection
# Which material to use from input list
MATERIAL_INDEX = 0

# Terrace parameters:
# Normal vector describing a plane that cuts the terrace from added layers (Miller indices)
CUT_DIRECTION = [0, 1, 1]
# Point the cutting plane passes through, in crystal coordinates
PIVOT_COORDINATE = [0.0, 0.0, 0.25]
# Height of terrace in atomic layers
NUMBER_OF_ADDED_LAYERS = 1
# Use cartesian instead of crystal coordinates
USE_CARTESIAN_COORDINATES = False
# Rotate to match periodic boundary conditions
ROTATE_TO_MATCH_PBC = True

# Slab parameters for creating a new slab if provided material is not a slab
SLAB_PARAMETERS = SimpleNamespace(
    miller_indices= (1, 1, 1),
    number_of_layers= 3,  # Number of layers in the slab
    thickness= 6,
    vacuum= 15.0,
    use_orthogonal_c= True,
    xy_supercell_matrix= [[2, 0], [0, 2]]
)

# Visualization parameters
SHOW_INTERMEDIATE_STEPS = True
# Structure repeat in view
CELL_REPETITIONS_FOR_VISUALIZATION = [1, 1, 1]

### 3.2. Configure a terrace defect

In [ ]:
from mat3ra.made.tools.helpers import create_terrace
from mat3ra.made.tools.modify import translate_to_z_level

slab_111 = create_slab(
    crystal=material,
    miller_indices=SLAB_PARAMETERS.miller_indices,
    number_of_layers=SLAB_PARAMETERS.number_of_layers,
    vacuum=SLAB_PARAMETERS.vacuum,
    use_orthogonal_c=SLAB_PARAMETERS.use_orthogonal_c,
    xy_supercell_matrix=SLAB_PARAMETERS.xy_supercell_matrix
)

terrace_slab = create_terrace(
    slab=slab_111,
    cut_direction=CUT_DIRECTION,
    pivot_coordinate=PIVOT_COORDINATE,
    number_of_added_layers=NUMBER_OF_ADDED_LAYERS,
    use_cartesian_coordinates=USE_CARTESIAN_COORDINATES,
    rotate_to_match_pbc=ROTATE_TO_MATCH_PBC
)
terrace_slab = translate_to_z_level(terrace_slab, "center")


### 3.3. Visualize the terrace defect

In [ ]:
visualize(terrace_slab, viewer="wave")

## 4. Add oxygen
### 4.1. Find an fcc hollow on Pt(111)
On Pt(111) the O2 precursor sits over an fcc hollow (top-fcc-bridge, FIG. 1. A): the site above a third-layer Pt, where the next Pt layer would go. On the 3-layer slab set in 3.1, `new_crystal_site` lands in the fcc hollow; with 2 or 4 layers it lands in the hcp or atop site.

In [ ]:
import numpy as np
from mat3ra.made.tools.helpers import AdatomDefectDict, create_multiple_adatom_defects

slab_111_with_oxygen_in_fcc_hollow = create_multiple_adatom_defects(
    slab=slab_111,
    defect_dicts=[
        AdatomDefectDict(
            element=ADSORBATE_ELEMENT, coordinate_2d=PRECURSOR_SITE_COORDINATE_111, distance_z=ADSORBATE_HEIGHT
        )
    ],
    placement_method="new_crystal_site",
)
slab_111_with_oxygen_in_fcc_hollow.to_cartesian()
fcc_hollow = np.array(slab_111_with_oxygen_in_fcc_hollow.coordinates_array[-1])[:2]

### 4.2. Molecule axis and the neighbouring hollow on Pt(111)
The molecule points from the hollow to its nearest top-layer Pt (the top site of FIG. 1. A); after dissociation the two O atoms sit in this hollow and the next fcc hollow, one Pt-Pt bond away on that side.

In [ ]:
from mat3ra.made.tools.helpers import get_atom_indices_by_layer

slab_111_cartesian = slab_111.clone()
slab_111_cartesian.to_cartesian()
top_layer_111 = np.array(slab_111_cartesian.coordinates_array)[get_atom_indices_by_layer(slab_111)[-1], :2]
top_platinum_by_distance = sorted(top_layer_111, key=lambda position: np.linalg.norm(position - fcc_hollow))
molecule_axis_111 = top_platinum_by_distance[0] - fcc_hollow
neighbor_vector_111 = top_platinum_by_distance[0] - top_platinum_by_distance[1]

### 4.3. Step edge and its bridge on Pt(211)
At the step the O2 precursor lies along the step edge, the topmost Pt row, centred over the bridge between two neighbouring edge atoms (top-bridge-top, FIG. 1. B).

In [ ]:
from itertools import combinations

slab_211_cartesian = slab_211.clone()
slab_211_cartesian.to_cartesian()
step_edge_211 = np.array(slab_211_cartesian.coordinates_array)[get_atom_indices_by_layer(slab_211)[-1], :2]
step_edge_pair = min(combinations(step_edge_211, 2), key=lambda pair: np.linalg.norm(pair[1] - pair[0]))
step_bridge, step_vector = sum(step_edge_pair) / 2, step_edge_pair[1] - step_edge_pair[0]

### 4.4. Build the precursors and the dissociated pairs
Each precursor is O2 centred on its site along the molecule axis, with the O-O distance of Table 1; each dissociated pair puts the two O atoms on the neighbouring sites (Pt(111): two fcc hollows, Pt(211): the step-edge bridges one Pt spacing to either side). All O atoms start at the same height above the topmost Pt for the relaxation. The first O of each precursor ends on the first site of its pair.

In [ ]:
half_bond_111 = molecule_axis_111 / np.linalg.norm(molecule_axis_111) * PRECURSOR_BOND_LENGTH_111 / 2
half_bond_211 = step_vector / np.linalg.norm(step_vector) * PRECURSOR_BOND_LENGTH_211 / 2
oxygen_positions = {
    "Pt(111) O2 precursor": (slab_111, [fcc_hollow + half_bond_111, fcc_hollow - half_bond_111]),
    "Pt(111) 2O dissociated": (slab_111, [fcc_hollow + neighbor_vector_111, fcc_hollow]),
    "Pt(211) step O2 precursor": (slab_211, [step_bridge + half_bond_211, step_bridge - half_bond_211]),
    "Pt(211) step 2O dissociated": (slab_211, [step_bridge + step_vector, step_bridge - step_vector]),
}
slabs_with_oxygen = []
for name, (slab, positions) in oxygen_positions.items():
    defect_dicts = [
        AdatomDefectDict(
            element=ADSORBATE_ELEMENT,
            coordinate_2d=position,
            distance_z=ADSORBATE_HEIGHT,
            use_cartesian_coordinates=True,
        )
        for position in positions
    ]
    slab_with_oxygen = create_multiple_adatom_defects(
        slab=slab, defect_dicts=defect_dicts, placement_method="exact_coordinate"
    )
    slab_with_oxygen = translate_to_z_level(slab_with_oxygen, "center")
    slab_with_oxygen.name = name
    slabs_with_oxygen.append(slab_with_oxygen)
    print(f"{name}: {slab_with_oxygen.basis.number_of_atoms} atoms")

### 4.5. Visualize the materials with oxygen

In [ ]:
visualize(slabs_with_oxygen, repetitions=VISUALIZATION_REPETITIONS, rotation="0x")
visualize(slabs_with_oxygen, repetitions=VISUALIZATION_REPETITIONS, rotation="-90x")

## 5. Download materials

In [ ]:
from mat3ra.notebooks_utils.io import download_content_to_file
from mat3ra.notebooks_utils.material import set_materials

slab_211 = translate_to_z_level(slab_211, "center")
set_materials([slab_211, terrace_slab, *slabs_with_oxygen])
download_content_to_file(slab_211.to_json(), "Pt_211_step.json")
download_content_to_file(terrace_slab.to_json(), "Pt_111_terrace.json")
for material in slabs_with_oxygen:
    download_content_to_file(material.to_json(), f"{material.name}.json")